In [2]:
import os
import time
import json
from datetime import datetime, timedelta
from typing import List, Dict
from zoneinfo import ZoneInfo

import requests
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from pymongo import MongoClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as spark_sum, month


In [6]:
import requests
import pandas as pd
from datetime import datetime
import time
from typing import List, Dict

# --- 1. CORE API FUNCTION (Generic) ---
def fetch_elhub_data(dataset_name: str, start_date: str, end_date: str) -> List[Dict]:
    base_url = "https://api.elhub.no/energy-data/v0/price-areas"
    params = {'dataset': dataset_name, 'startDate': start_date, 'endDate': end_date}

    # Convert UPPER_CASE dataset name to camelCase key for JSON parsing
    # e.g. PRODUCTION_PER_GROUP_MBA_HOUR -> productionPerGroupMbaHour
    tokens = dataset_name.lower().split('_')
    json_key = tokens[0] + ''.join(x.title() for x in tokens[1:])

    try:
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        all_records = []
        if 'data' in data:
            for price_area_data in data['data']:
                if 'attributes' in price_area_data and json_key in price_area_data['attributes']:
                    all_records.extend(price_area_data['attributes'][json_key])
        
        return all_records

    except requests.exceptions.RequestException as e:
        # If a specific month fails, we print error but don't crash the whole script
        print(f"  [!] Error fetching {start_date[:7]}: {e}")
        return []

# --- 2. LOOPING FUNCTION (Years + Months) ---
def fetch_data_monthly_loop(dataset_name: str, start_year: int, end_year: int) -> pd.DataFrame:
    
    all_records = []
    
    print(f"Starting retrieval for: {dataset_name}")
    print(f"Period: {start_year} - {end_year}")

    # Loop through every year requested
    for year in range(start_year, end_year + 1):
        print(f"\n  Processing Year: {year}")
        
        # Loop through every month (Safety against 400 Bad Request)
        for month in range(1, 13):
            
            # Calculate start and end of the month
            month_start = datetime(year, month, 1, 0, 0, 0)
            
            # Logic for rollover (Dec -> Jan)
            if month == 12:
                month_end = datetime(year + 1, 1, 1, 0, 0, 0)
            else:
                month_end = datetime(year, month + 1, 1, 0, 0, 0)

            # Format for API
            start_str = month_start.strftime('%Y-%m-%dT%H:%M:%S+01:00')
            end_str = month_end.strftime('%Y-%m-%dT%H:%M:%S+01:00')

            # Fetch
            records = fetch_elhub_data(dataset_name, start_str, end_str)
            
            if records:
                all_records.extend(records)
                print(f"    - {month_start.strftime('%B')}: Retrieved {len(records)} records")
            else:
                print(f"    - {month_start.strftime('%B')}: No data")

            # Sleep to avoid "429 Too Many Requests"
            time.sleep(0.2)

    # Convert to DataFrame
    df = pd.DataFrame(all_records)

    # Fix Timezones
    if not df.empty:
        print(f"\n  Formatting timezones for {len(df)} rows...")
        cols = ['startTime', 'endTime', 'lastUpdatedTime']
        for col in cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], utc=True).dt.tz_convert("Europe/Oslo")
    
    return df

# --- 3. MAIN EXECUTION ---
if __name__ == "__main__":
    
    # --- PART 1: PRODUCTION (2022 - 2024) ---
    # We skip 2021 because you already have it from the previous assignment part
    print(">>> TASK 1: FETCHING PRODUCTION DATA (2022-2024)")
    df_prod = fetch_data_monthly_loop(
        dataset_name="PRODUCTION_PER_GROUP_MBA_HOUR", 
        start_year=2022, 
        end_year=2024
    )
    
    if not df_prod.empty:
        filename = "elhub_production_2022_2024.csv"
        df_prod.to_csv(filename, index=False)
        print(f"SUCCESS: Saved {filename}")
    
    print("-" * 40)

    # --- PART 2: CONSUMPTION (2021 - 2024) ---
    # Assignment asks for full range 2021-2024 for consumption
    print(">>> TASK 2: FETCHING CONSUMPTION DATA (2021-2024)")
    df_cons = fetch_data_monthly_loop(
        dataset_name="CONSUMPTION_PER_GROUP_MBA_HOUR", 
        start_year=2021, 
        end_year=2024
    )
    
    if not df_cons.empty:
        filename = "elhub_consumption_2021_2024.csv"
        df_cons.to_csv(filename, index=False)
        print(f"SUCCESS: Saved {filename}")

    print("\nAll operations complete.")

>>> TASK 1: FETCHING PRODUCTION DATA (2022-2024)
Starting retrieval for: PRODUCTION_PER_GROUP_MBA_HOUR
Period: 2022 - 2024

  Processing Year: 2022
    - January: Retrieved 18600 records
    - February: Retrieved 16800 records
    - March: Retrieved 18575 records
    - April: Retrieved 18000 records
    - May: Retrieved 18600 records
    - June: Retrieved 18000 records
    - July: Retrieved 18600 records
    - August: Retrieved 18600 records
    - September: Retrieved 18000 records
    - October: Retrieved 18625 records
    - November: Retrieved 18000 records
    - December: Retrieved 18600 records

  Processing Year: 2023
    - January: Retrieved 18600 records
    - February: Retrieved 16800 records
    - March: Retrieved 18575 records
    - April: Retrieved 18000 records
    - May: Retrieved 18600 records
    - June: Retrieved 18000 records
    - July: Retrieved 18600 records
    - August: Retrieved 18600 records
    - September: Retrieved 18000 records
    - October: Retrieved 18625